# E2a — Perovskite-specialized ChargeFlow (Colab, resumable)

**Goal:** chemistry-homogeneous proof-of-principle — `<1%` ε_MAE on a parent-disjoint perovskite split.

**Runtime:** Runtime → Change runtime type → **T4 GPU (free tier)**. Do NOT pay for A100/L4.

> Each perovskite has a different 3D grid size, so training uses `batch_size=1`
> with `accum_iter=16` for an effective batch of 16. The model is tiny
> (`unet_cond`, model_channels=4) — well under T4's 16 GB even at native resolution.

**Estimated cost:** $0 — free Colab T4 is sufficient for all 300 epochs.

## 0. Verify GPU

In [69]:
!nvidia-smi


Sun Jun 21 04:05:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive

In [70]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/chargeflow'
DATA_DIR   = f'{DRIVE_ROOT}/data_perovskite'
OUTPUT_DIR = f'{DRIVE_ROOT}/output_perovskite'
LISTS_DIR  = f'{DRIVE_ROOT}/lists_perovskite'
for d in (DRIVE_ROOT, DATA_DIR, OUTPUT_DIR, LISTS_DIR):
    os.makedirs(d, exist_ok=True)
print('Drive ready:', DRIVE_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive ready: /content/drive/MyDrive/chargeflow


## 2. Clone repo

In [71]:
import os, shutil
REPO_DIR = '/content/chargeflow'

%cd /content

# Remove stale clone if present
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('Removed old clone.')

!git clone https://github.com/ngminhtri0394/chargeflow-electron-density-main.git {REPO_DIR}

# The GitHub repo may have a nested subfolder — find where scripts/ actually lives
actual_root = REPO_DIR
for entry in os.listdir(REPO_DIR):
    candidate = f'{REPO_DIR}/{entry}'
    if os.path.isdir(f'{candidate}/scripts'):
        actual_root = candidate
        print(f'Found scripts/ inside subfolder: {entry}')
        break
REPO_DIR = actual_root

%cd {REPO_DIR}
assert os.path.isdir(f'{REPO_DIR}/scripts'), f'scripts/ not found under {REPO_DIR}, contents: {os.listdir(REPO_DIR)}'
print('Done. Repo ready at', REPO_DIR)


/content
Removed old clone.
Cloning into '/content/chargeflow'...
remote: Enumerating objects: 210, done.
remote: Counting objects: 100% (210/210), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 210 (delta 102), reused 204 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (210/210), 798.48 KiB | 3.26 MiB/s, done.
Resolving deltas: 100% (102/102), done.
Found scripts/ inside subfolder: chargeflow-electron-density-main
/content/chargeflow/chargeflow-electron-density-main
Done. Repo ready at /content/chargeflow/chargeflow-electron-density-main


In [72]:
# Install all deps: use pip install -e . first (picks up setup.py),
# then add packages not listed in requirements.txt that src/ actually imports.
!pip -q install -e .
!pip -q install flow_matching ase pymatgen torchmetrics torchvision pyyaml
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


  Preparing metadata (setup.py) ... done
torch 2.11.0+cu128 cuda True


## 3. Write config (embedded — no upload needed)

The config lives directly in this cell so it works even on a fresh clone that doesn't have `config/train_perovskite_specialized.yaml`.

In [73]:
import yaml, os

cfg = {
    'model': {
        'architecture': 'unet_cond',
        'discrete_flow_matching': False,
        'use_ema': True,
    },
    'data': {
        'train_data_list':      f'{LISTS_DIR}/list_d_perovskite_train',
        'train_label_list':     f'{LISTS_DIR}/list_l_perovskite_train',
        'train_data_gridsize':  f'{LISTS_DIR}/list_dgs_perovskite_train',
        'train_label_gridsize': f'{LISTS_DIR}/list_lgs_perovskite_train',
        'use_charge_dataset':   True,
        'data_augmentation':    True,
        'downsample_data':      1,
        'downsample_label':     1,
        'normalize_density':    False,
        # batch_size MUST be 1: each perovskite has a different 3D grid size,
        # so densities cannot be stacked into a batch (zero-padding would be
        # physically wrong). Effective batch comes from accum_iter below.
        'batch_size':           1,
        'num_workers':          2,
        'pin_memory':           True,
    },
    'training': {
        'epochs':           300,
        'start_epoch':      0,
        'optimizer':        'AdamW',
        'learning_rate':    1e-4,
        'optimizer_betas':  [0.9, 0.95],
        'decay_lr':         True,
        'accum_iter':       16,  # effective batch = 1 x 16 = 16
        'class_drop_prob':  0.1,
        # save_frequency in epochs. After your first epoch, check the log for epoch time
        # and set this so checkpoints land every ~15 min: ceil(900s / epoch_time_seconds)
        'save_frequency':   10,
        'save_best':        True,
        'log_frequency':    10,
    },
    'evaluation': {
        'eval_frequency': 50,
        'compute_fid':    False,
        'save_samples':   True,
    },
    'ode': {
        'method':           'dopri5',
        'options':          {'atol': 1e-5, 'rtol': 1e-5},
        'cfg_scale':        1.0,
        'sampling_dtype':   'float32',
        'skewed_timesteps': True,
        'edm_schedule':     True,
    },
    'distributed': {'enabled': False, 'backend': 'nccl', 'init_method': 'env://'},
    'output': {
        'output_dir':   OUTPUT_DIR,
        'save_postfix': 'Perovskite_E2a',
        'log_file':     'training_perovskite.log',
    },
    'seed':      42,
    'device':    'cuda',
    'test_run':  False,
    'eval_only': False,
}

RUN_CFG = f'{OUTPUT_DIR}/run_config.yaml'
with open(RUN_CFG, 'w') as fh:
    yaml.safe_dump(cfg, fh, sort_keys=False)
print('Config written to', RUN_CFG)


Config written to /content/drive/MyDrive/chargeflow/output_perovskite/run_config.yaml


## 4. Download + extract perovskite data (ONCE — lives on Drive)

Skip on later sessions — data is already on Drive if the marker file exists.

In [74]:
import os
ZENODO = 'https://zenodo.org/records/19211405/files'
files = [
    'perovskites_inputs_initial_density.tar.gz',
    'perovskites_targets_dft_density.tar.gz',
    'external_periodic_test_perovskites_split.csv',
    'paper_splits_all.csv',
]
marker = f'{DATA_DIR}/.extracted'
if not os.path.exists(marker):
    for f in files:
        local = f'/content/{f}'
        !wget -q -c "{ZENODO}/{f}" -O "{local}"
        if f.endswith('.tar.gz'):
            !tar -xzf "{local}" -C "{DATA_DIR}"
        else:
            !cp "{local}" "{DATA_DIR}/"
    open(marker, 'w').close()
    print('Extracted to', DATA_DIR)
else:
    print('Already extracted — skipping.')


Already extracted — skipping.


## 5. Inspect extracted filenames

Run once to confirm folder/file structure before building splits.

In [75]:
import glob, os
npys = glob.glob(f'{DATA_DIR}/**/*.npy', recursive=True)
print('total .npy:', len(npys))
print('--- first 10 ---')
for p in sorted(npys)[:10]:
    print(p)
# show top-level subdirs
print('\n--- top-level subdirs ---')
for d in sorted(os.listdir(DATA_DIR)):
    full = f'{DATA_DIR}/{d}'
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f'  {d}/  ({n} entries)')


total .npy: 318
--- first 10 ---
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_-1/rho_22.npy
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_-2/rho_22.npy
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_1/rho_22.npy
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_2/rho_22.npy
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1183053_Ac1Ga1O3_remove_24_charge_-1/rho_22.npy
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1183053_Ac1Ga1O3_remove_24_charge_-2/rho_22.npy
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1183053_Ac1Ga1O3_remove_24_charge_1/rho_22.npy
/content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho/VASP_mp-1183053_Ac1Ga1O3_remove_24_charge_2/rho_22

## 6. Build parent-disjoint train/test split + dataset file lists

Based on actual filenames from Colab output:
```
perovskites_rho_initial/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_-1/rho_22.npy
```
- Inputs folder: `perovskites_rho_initial/`
- Targets folder: `perovskites_rho_dft/` (or similar — confirmed by the inspect cell above)
- Pairing key: subfolder name `VASP_mp-XXXXX_..._charge_N`
- Parent id: `mp-XXXXX` (first MP id in the subfolder name)

The gridsize is recovered from the `.npy` array shape (written as a sidecar `.txt`).

In [76]:
import re, glob, random, os
import numpy as np
random.seed(42)

# --- identify input and target root dirs ---
# 'perovskites_rho_initial' = SAD/neutral inputs ; 'perovskites_rho' = DFT targets
subdirs = [d for d in os.listdir(DATA_DIR)
           if os.path.isdir(f'{DATA_DIR}/{d}') and d.startswith('perovskites')]
print('perovskite subdirs:', subdirs)

INPUT_SUBDIR  = next((d for d in subdirs if 'initial' in d or 'input' in d), None)
TARGET_SUBDIR = next((d for d in subdirs if d != INPUT_SUBDIR), None)
assert INPUT_SUBDIR and TARGET_SUBDIR, f'Cannot resolve input/target in {subdirs}'
print('inputs :', INPUT_SUBDIR, '\ntargets:', TARGET_SUBDIR)

INPUT_ROOT  = f'{DATA_DIR}/{INPUT_SUBDIR}'
TARGET_ROOT = f'{DATA_DIR}/{TARGET_SUBDIR}'

# --- pair input/target by matching subfolder name ---
def find_npy(root_dir):
    result = {}
    for subdir in os.listdir(root_dir):
        p = f'{root_dir}/{subdir}'
        if os.path.isdir(p):
            npys = glob.glob(f'{p}/*.npy')
            if npys:
                result[subdir] = npys[0]
    return result

inp_map = find_npy(INPUT_ROOT)
tgt_map = find_npy(TARGET_ROOT)
common  = sorted(set(inp_map) & set(tgt_map))
pairs   = [(inp_map[k], tgt_map[k]) for k in common]
print(f'paired: {len(pairs)}')

def parent_id_of(subdir_name):
    m = re.search(r'(mp-\d+)', subdir_name)
    return m.group(1) if m else subdir_name

# --- gridsize sidecar from the REAL .npy shape (arrays are already 3D) ---
# The dataset does np.load(npy).reshape(1, *gridsize), so gridsize MUST equal
# the true array shape. We read it directly — no guessing.
def gridsize_path_for(npy_path):
    txt = npy_path.replace('.npy', '.realshape.txt')
    arr = np.load(npy_path, mmap_mode='r')
    assert arr.ndim == 3, f'Expected 3D array, got {arr.ndim}D for {npy_path}'
    with open(txt, 'w') as fh:
        fh.write(' '.join(map(str, arr.shape)))
    return txt

# --- parent-disjoint 85/15 split ---
parents  = sorted({parent_id_of(k) for k in common})
random.shuffle(parents)
n_test   = max(1, int(0.15 * len(parents)))
test_set = set(parents[:n_test])
train_pairs = [(i, t) for k, (i, t) in zip(common, pairs) if parent_id_of(k) not in test_set]
test_pairs  = [(i, t) for k, (i, t) in zip(common, pairs) if parent_id_of(k) in test_set]
print(f'parents: {len(parents)}  train: {len(train_pairs)}  test: {len(test_pairs)}')

def write_lists(split_pairs, tag):
    data_paths  = [p[0] for p in split_pairs]
    label_paths = [p[1] for p in split_pairs]
    data_gs     = [gridsize_path_for(p) for p in data_paths]
    label_gs    = [gridsize_path_for(p) for p in label_paths]
    for name, items in [
        (f'list_d_{tag}',   data_paths),
        (f'list_l_{tag}',   label_paths),
        (f'list_dgs_{tag}', data_gs),
        (f'list_lgs_{tag}', label_gs),
    ]:
        with open(f'{LISTS_DIR}/{name}', 'w') as fh:
            fh.write('\n'.join(items) + '\n')
    print(f'wrote {tag} lists ({len(split_pairs)} samples)')

write_lists(train_pairs, 'perovskite_train')
write_lists(test_pairs,  'perovskite_test')

# sanity check: confirm a written gridsize matches the npy shape
chk_npy = data_paths_check = open(f'{LISTS_DIR}/list_d_perovskite_train').readline().strip()
chk_gs  = open(f'{LISTS_DIR}/list_dgs_perovskite_train').readline().strip()
print('\nSanity:', np.load(chk_npy, mmap_mode='r').shape, '==', open(chk_gs).read())


perovskite subdirs: ['perovskites_rho_initial', 'perovskites_rho']
inputs : perovskites_rho_initial 
targets: perovskites_rho
paired: 159
parents: 40  train: 135  test: 24
wrote perovskite_train lists (135 samples)
wrote perovskite_test lists (24 samples)

Sanity: (96, 140, 140) == 96 140 140


In [77]:
# DIAGNOSTIC: check the actual shape of the .npy density files and whether
# the original grid_sizes_*.dat files were shipped with the data.
import glob, os, numpy as np

sample_dir = glob.glob(f'{DATA_DIR}/perovskites_rho_initial/*/')[0]
print('Sample structure folder:', sample_dir)
print('Files in it:', os.listdir(sample_dir))
print()

npy = glob.glob(f'{sample_dir}/*.npy')[0]
arr = np.load(npy)
print(f'{os.path.basename(npy)}: shape={arr.shape}, ndim={arr.ndim}, size={arr.size}')
if arr.ndim == 1:
    cube = round(arr.size ** (1/3))
    print(f'  flat array. cube-root = {cube} (cube={cube**3==arr.size})')
    print('  -> need the real grid_sizes_*.dat to reshape correctly!')

# Check a few more structures to see if shapes/sizes differ across the family
print('\nSizes across 8 structures:')
for d in sorted(glob.glob(f'{DATA_DIR}/perovskites_rho_initial/*/'))[:8]:
    p = glob.glob(f'{d}/*.npy')[0]
    a = np.load(p, mmap_mode='r')
    print(f'  {os.path.basename(d.rstrip("/"))[:40]:40s} shape={a.shape} size={a.size}')


Sample structure folder: /content/drive/MyDrive/chargeflow/data_perovskite/perovskites_rho_initial/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_-1/
Files in it: ['rho_22.npy', 'grid_sizes_22.dat', 'rho_22.gridsize.txt', 'rho_22.realshape.txt']

rho_22.npy: shape=(96, 140, 140), ndim=3, size=1881600

Sizes across 8 structures:
  VASP_mp-1055491_Rb1Ag1F3_remove_23_charg shape=(96, 140, 140) size=1881600
  VASP_mp-1055491_Rb1Ag1F3_remove_23_charg shape=(96, 140, 140) size=1881600
  VASP_mp-1055491_Rb1Ag1F3_remove_23_charg shape=(96, 140, 140) size=1881600
  VASP_mp-1055491_Rb1Ag1F3_remove_23_charg shape=(96, 140, 140) size=1881600
  VASP_mp-1183053_Ac1Ga1O3_remove_24_charg shape=(84, 84, 84) size=592704
  VASP_mp-1183053_Ac1Ga1O3_remove_24_charg shape=(84, 84, 84) size=592704
  VASP_mp-1183053_Ac1Ga1O3_remove_24_charg shape=(84, 84, 84) size=592704
  VASP_mp-1183053_Ac1Ga1O3_remove_24_charg shape=(84, 84, 84) size=592704


## 7. Find latest checkpoint (auto-resume)

Run this at the start of every session — it picks up where the last session left off.

In [78]:
# Pull latest code fixes (REPO_DIR is the nested path resolved in cell 2)
!git -C {REPO_DIR} pull


Already up to date.


In [79]:
!git -C /content/chargeflow pull


Already up to date.


## 8. Train (resumable)

Runs until `epochs` or until you stop the cell. Each disconnect is safe — the last Drive checkpoint is intact.

**After your first epoch completes**, note the epoch time printed in the log and adjust `save_frequency` in the config dict above (cell 3) so checkpoints land every ~15 min, then re-run cell 3 + this cell.

In [80]:
resume_arg = f'--resume "{RESUME}"' if RESUME else ''
!cd {REPO_DIR} && python scripts/train.py --config "{RUN_CFG}" {resume_arg}


2026-06-21 04:06:03 [INFO    ] train: ================================================================================
2026-06-21 04:06:03 [INFO    ] train: Starting Training
2026-06-21 04:06:03 [INFO    ] train: ================================================================================
2026-06-21 04:06:03 [INFO    ] train: Output directory: /content/drive/MyDrive/chargeflow/output_perovskite
2026-06-21 04:06:03 [INFO    ] train: Saved configuration to /content/drive/MyDrive/chargeflow/output_perovskite/config.yaml
Not using distributed mode
2026-06-21 04:06:04 [INFO    ] train: Device: cuda
2026-06-21 04:06:04 [INFO    ] train: Seed: 42
2026-06-21 04:06:04 [INFO    ] train: Distributed: False
2026-06-21 04:06:04 [INFO    ] train: Creating dataset (charge-aware: True)
2026-06-21 04:06:04 [INFO    ] train: Training dataset size: 135
2026-06-21 04:06:04 [INFO    ] train: Creating model...
2026-06-21 04:06:04 [INFO    ] train: Model architecture: unet_cond
2026-06-21 04:06:04 [INFO 

## 9. Final evaluation — the editor-gate ε_MAE number

Run after training converges. Uses the parent-disjoint test split. Target: **< 0.01 (1%)**.

In [81]:
import sys; sys.path.insert(0, REPO_DIR)
import numpy as np, torch, statistics
from src.utils.metrics import calculate_normalized_mae

test_d = open(f'{LISTS_DIR}/list_d_perovskite_test').read().split()
test_l = open(f'{LISTS_DIR}/list_l_perovskite_test').read().split()
print(f'{len(test_d)} held-out test structures (parent-disjoint)')

# TODO: wire in the inference path.
# The repo eval loop lives in src/training/eval_loop.py / scripts/predict.py.
# Replace the stub below with the actual ODE-sampled predictions.
#
# Suggested approach:
#   !cd {REPO_DIR} && python scripts/predict.py \
#       --config "{RUN_CFG}" \
#       --checkpoint "{OUTPUT_DIR}/best_model-Perovskite_E2a.pth" \
#       --data-list  "{LISTS_DIR}/list_d_perovskite_test" \
#       --label-list "{LISTS_DIR}/list_l_perovskite_test" \
#       --nfe 100 \
#       --output-dir "{OUTPUT_DIR}/test_preds"
#
# Then load the saved predictions and compute:
eps = []
# for pred_path, tgt_path in zip(sorted_pred_paths, test_l):
#     pred   = torch.tensor(np.load(pred_path), dtype=torch.float32)
#     target = torch.tensor(np.load(tgt_path),  dtype=torch.float32)
#     eps.append(calculate_normalized_mae(pred, target).item())
#
# if eps:
#     print(f'mean ε_MAE : {statistics.mean(eps)*100:.3f}%')
#     print(f'median     : {statistics.median(eps)*100:.3f}%')
#     print(f'frac < 1%  : {sum(e<0.01 for e in eps)/len(eps)*100:.1f}%')

print('Stub — wire in scripts/predict.py to fill eps[].')


24 held-out test structures (parent-disjoint)
Stub — wire in scripts/predict.py to fill eps[].
